# Part 4
In this part we will make our model more in-line with our submission-format, mainly by transforming to cumulative weights.


In [1]:
#Imports
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#Our dataframe
final_df = pd.read_csv("cleaned_data/cleaned_df_final_test.csv") #TEMP NAME, remember to change.
#Variables that we need

In [2]:
#Define what proportion should be reserved for verifying. Evaluation will be done with the latter part.
SPLIT_RATIO = 0.975

#Find what day this corresponds to
split_day_numerical = final_df["days_normalized"].quantile(SPLIT_RATIO)
print("Splitting at ", SPLIT_RATIO, ", Corresponds to day", split_day_numerical)

#Split data according to our index
train_df = final_df[final_df["days_normalized"] <= split_day_numerical]
test_df = final_df[final_df["days_normalized"] > split_day_numerical]
#Verify the split

print(f"Train range: {train_df['days_normalized'].min()} to {train_df['days_normalized'].max()}")
print(f"Test range: {test_df['days_normalized'].min()} to {test_df['days_normalized'].max()}")

first_test_date = test_df["date_str"].iloc[0]
last_test_date = test_df["date_str"].iloc[-1]
print("The data used for testings starts at ", first_test_date, ", and ends at ", last_test_date)


Splitting at  0.975 , Corresponds to day 7305.0
Train range: 0 to 7305
Test range: 7306 to 7492
The data used for testings starts at  2024-06-16 , and ends at  2024-12-19


Create a cumulative sum for test data to verify the training data.


In [3]:
valid_ids = final_df["rm_id"].unique()

cumsums = []
for id in valid_ids:
    temp_df = test_df[test_df["rm_id"] == id]
    cumsum = temp_df["daily_weight"].cumsum()
    cumsums.extend(cumsum.to_list())


test_df["cumulative_weight"] = cumsums



C:\Users\aksel\AppData\Local\Temp\ipykernel_5720\3157555919.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["cumulative_weight"] = cumsums


Convert predictions to cumsums as well:

In [ ]:
#test_predictions = pd.read_csv("test_predictions2.csv")
test_predictions = pd.read_csv("test_predictions_simple.csv")

In [16]:
cumsums = []
for id in valid_ids:
    temp_df = test_predictions[test_predictions["rm_id"] == id]
    cumsum = temp_df["predicted_weight"].cumsum()
    cumsums.extend(cumsum.to_list())


test_predictions["cumulative_weight"] = cumsums

Evaluate performance:

In [17]:
predictions = test_predictions["cumulative_weight"]
actual = test_df["cumulative_weight"]

mse = mean_squared_error(predictions, actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(predictions, actual)
r2 = r2_score(predictions, actual)


print(f'Mean squared Error: {mse:.4f}')
print(f'Root Mean Squared: {rmse:.4f}')
print(f'Mean absolute error: {mae:.4f}')
print(f'r2-score: {r2:.4f}')

Mean squared Error: 315038569.1344
Root Mean Squared: 17749.3259
Mean absolute error: 3789.6647
r2-score: 0.9990
